In [0]:
from pyspark.sql import functions as f

In [0]:
# Define Source path
SOURCE_PATH = "/Volumes/urban-mobility-data-platform-dev/source/data/"

In [0]:
# Read Data
df = spark.read.parquet(f"{SOURCE_PATH}/yellow_tripdata_2026-*.parquet")
print("Rows", df.count())
print("Colums", len(df.columns))

In [0]:
# Display Data
display(df.limit(100))

In [0]:
# Verify all months (Jan,Feb,Mar,Apr) Data is in DF(combined)
df.groupBy(f.month(f.col("tpep_dropoff_datetime")).alias("month")).agg(f.count("*").alias('count')).orderBy("month").display()

In [0]:
# Add metadata fields to this Source DF to turn it into Bronze DF
bronze_df = df.withColumn("_source_file",f.col("_metadata.file_path"))\
              .withColumn("_source_month", f.regexp_extract(
                        f.col("_source_file"),
                        r"yellow_tripdata_(\d{4}-\d{2})\.parquet",
                        1
                        ))\
              .withColumn("_ingested_at", f.current_timestamp())\
              .withColumn("_ingestion_batch_id", f.lit("initial_load_2026_04"))

In [0]:
display(bronze_df.filter(f.col("_source_month") == "2026-01").limit(5))
# January

In [0]:
display(bronze_df.filter(f.col("_source_month") == "2026-02").limit(5))
# February

In [0]:
display(bronze_df.filter(f.col("_source_month") == "2026-03").limit(5))
# March

In [0]:
display(bronze_df.filter(f.col("_source_month") == "2026-04").limit(5))
# March

In [0]:
# Create Hash Key column
record_columns = [
    "VendorID",
    "tpep_pickup_datetime",
    "tpep_dropoff_datetime",
    "passenger_count",
    "trip_distance",
    "RatecodeID",
    "store_and_fwd_flag",
    "PULocationID",
    "DOLocationID",
    "payment_type",
    "fare_amount",
    "extra",
    "mta_tax",
    "tip_amount",
    "tolls_amount",
    "improvement_surcharge",
    "total_amount",
    "congestion_surcharge",
    "Airport_fee",
    "cbd_congestion_fee"
]

bronze_df = bronze_df.withColumn("_record_hash", f.sha2(f.to_json(f.struct(*[f.col(c) for c in record_columns])), 256))

In [0]:
bronze_df.printSchema()

In [0]:
display(bronze_df.limit(10))

In [0]:
# Create Table From bronze_df
bronze_df.write.format("delta").mode("overwrite").saveAsTable("`urban-mobility-data-platform-dev`.bronze.yellow_trips_raw")

In [0]:
%sql
use `urban-mobility-data-platform-dev`.bronze

In [0]:
%sql
SELECT * from bronze.yellow_trips_raw;

In [0]:
%sql
SELECT COUNT(*) from bronze.yellow_trips_raw;

In [0]:
%sql
DESC bronze.yellow_trips_raw;